# KIRA Plant Disease Deep Learning (Kaggle)

In [ ]:
!pip -q install torch torchvision timm scikit-learn matplotlib seaborn pillow

In [ ]:
import os, glob, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from sklearn.metrics import classification_report, confusion_matrix
print('torch', torch.__version__)

## 1. Dataset & EDA

In [ ]:
DATA_DIR = '/kaggle/input/new-plant-diseases-dataset'  # sesuaikan
train_dir = os.path.join(DATA_DIR,'New Plant Diseases Dataset(Augmented)','New Plant Diseases Dataset(Augmented)','train')
val_dir = os.path.join(DATA_DIR,'New Plant Diseases Dataset(Augmented)','New Plant Diseases Dataset(Augmented)','valid')
print('train exists', os.path.exists(train_dir))
print('val exists', os.path.exists(val_dir))

In [ ]:
train_tf = transforms.Compose([transforms.Resize((224,224)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(10), transforms.ToTensor()])
val_tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor()])
train_ds = datasets.ImageFolder(train_dir, transform=train_tf)
val_ds = datasets.ImageFolder(val_dir, transform=val_tf)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
print('classes', len(train_ds.classes))
print('train size', len(train_ds), 'val size', len(val_ds))

In [ ]:
# Visualisasi distribusi kelas
import collections
counts = collections.Counter([y for _,y in train_ds.samples])
names = [train_ds.classes[i] for i in counts.keys()]
plt.figure(figsize=(12,6))
plt.bar(range(len(counts)), list(counts.values()))
plt.xticks(range(len(counts)), names, rotation=90, fontsize=7)
plt.title('Distribusi Kelas (Train)')
plt.tight_layout()
plt.show()

In [ ]:
# Tampilkan sample gambar
fig, axes = plt.subplots(2,5, figsize=(12,5))
for ax in axes.flatten():
    img, label = train_ds[random.randint(0,len(train_ds)-1)]
    ax.imshow(img.permute(1,2,0))
    ax.set_title(train_ds.classes[label], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images')
plt.tight_layout()
plt.show()

## 2. Model (MobileNetV3)

In [ ]:
model = timm.create_model('mobilenetv3_large_100', pretrained=True, num_classes=len(train_ds.classes))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
print('device', device)

## 3. Training

In [ ]:
history = {'train_loss':[], 'val_acc':[]}
for epoch in range(5):
    model.train()
    running=0
    for x,y in train_loader:
        x,y = x.to(device), y.to(device)
        opt.zero_grad()
        out = model(x)
        loss = crit(out,y)
        loss.backward()
        opt.step()
        running += loss.item()
    # val
    model.eval()
    correct=0; total=0
    with torch.no_grad():
        for x,y in val_loader:
            x,y=x.to(device),y.to(device)
            pred=model(x).argmax(1)
            correct += (pred==y).sum().item()
            total += y.size(0)
    val_acc = correct/total
    history['train_loss'].append(running/len(train_loader))
    history['val_acc'].append(val_acc)
    print(f'epoch {epoch} loss {running/len(train_loader):.4f} val_acc {val_acc:.4f}')

In [ ]:
# Plot training curve
fig, ax = plt.subplots(1,2, figsize=(10,4))
ax[0].plot(history['train_loss'], marker='o')
ax[0].set_title('Train Loss')
ax[1].plot(history['val_acc'], marker='o', color='green')
ax[1].set_title('Val Accuracy')
plt.tight_layout()
plt.show()

## 4. Evaluasi

In [ ]:
model.eval()
y_true=[]; y_pred=[]
with torch.no_grad():
    for x,y in val_loader:
        x=x.to(device)
        pred=model(x).argmax(1).cpu().numpy()
        y_pred.extend(pred)
        y_true.extend(y.numpy())
print(classification_report(y_true, y_pred, target_names=train_ds.classes, digits=3))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12,10))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=train_ds.classes, yticklabels=train_ds.classes)
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 5. Simpan Model

In [ ]:
torch.save(model.state_dict(), '/kaggle/working/kira_plant_model.pth')
print('saved')